# SQL Database Agent @ `LangGraph`

## Creating & Populating Database

In [39]:
import sqlite3

In [40]:
# Connect to SQLite database

connection = sqlite3.connect('mydb.db')

In [41]:
connection

In [42]:
# Creating table(s)

table_create_query_1 = """
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    prhire_date TEXT NOT NULL,
    salary INTEGER NOT NULL
);
"""

In [43]:
table_create_query_2 = """
CREATE TABLE IF NOT EXISTS customers (
  customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
  first_name TEXT NOT NULL,
  last_name TEXT NOT NULL,
  email TEXT UNIQUE NOT NULL,
  phone TEXT
);
"""

In [44]:
table_create_query_3 = """
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);"""

In [45]:
# Creating cursor object

cursor = connection.cursor()

In [46]:
# Passing table creation queries to the database

cursor.execute(table_create_query_1)
cursor.execute(table_create_query_2)
cursor.execute(table_create_query_3)

In [47]:
# Dummy Data for Employees
employees_data = [
    (1, 'Alice', 'Johnson', 'alice.j@example.com', '2023-01-15', 75000),
    (2, 'Bob', 'Smith', 'bob.s@example.com', '2023-03-20', 68000),
    (3, 'Charlie', 'Davis', 'charlie.d@example.com', '2022-11-05', 82000),
    (4, 'Diana', 'Prince', 'diana.p@example.com', '2024-02-10', 95000),
    (5, 'Edward', 'Norton', 'edward.n@example.com', '2023-06-30', 70000)
]

# Dummy Data for Customers
customers_data = [
    ('John', 'Doe', 'john.doe@email.com', '555-0101'),
    ('Jane', 'Smith', 'jane.smith@email.com', '555-0102'),
    ('Michael', 'Brown', 'michael.b@email.com', '555-0103'),
    ('Emily', 'Davis', 'emily.d@email.com', '555-0104'),
    ('Chris', 'Wilson', 'chris.w@email.com', '555-0105')
]

# Dummy Data for Orders
import datetime
orders_data = [
    (1, datetime.date(2024, 1, 10).isoformat(), 150.50),
    (2, datetime.date(2024, 1, 12).isoformat(), 200.00),
    (1, datetime.date(2024, 2, 5).isoformat(), 45.75),
    (3, datetime.date(2024, 2, 14).isoformat(), 320.00),
    (4, datetime.date(2024, 3, 1).isoformat(), 12.99),
    (5, datetime.date(2024, 3, 5).isoformat(), 89.50),
    (2, datetime.date(2024, 3, 10).isoformat(), 110.25),
    (3, datetime.date(2024, 3, 15).isoformat(), 55.00),
    (1, datetime.date(2024, 3, 20).isoformat(), 250.00),
    (4, datetime.date(2024, 3, 22).isoformat(), 75.60)
]

cursor = connection.cursor()

# Insert Employees
cursor.executemany("INSERT OR IGNORE INTO employees (emp_id, first_name, last_name, email, prhire_date, salary) VALUES (?, ?, ?, ?, ?, ?)", employees_data)

# Insert Customers (using NULL for auto-increment ID)
cursor.executemany("INSERT OR IGNORE INTO customers (first_name, last_name, email, phone) VALUES (?, ?, ?, ?)", customers_data)

# Insert Orders
cursor.executemany("INSERT OR IGNORE INTO orders (customer_id, order_date, amount) VALUES (?, ?, ?)", orders_data)

connection.commit()
print("Dummy data populated successfully!")


Dummy data populated successfully!


In [48]:
# Retrieving data from table "orders"
cursor.execute("SELECT * FROM orders;")

In [49]:
for i in cursor.fetchall(): # Rows from table "orders"
    print(i)

(1, 1, '2024-01-10', 150.5)
(2, 2, '2024-01-12', 200.0)
(3, 1, '2024-02-05', 45.75)
(4, 3, '2024-02-14', 320.0)
(5, 4, '2024-03-01', 12.99)
(6, 5, '2024-03-05', 89.5)
(7, 2, '2024-03-10', 110.25)
(8, 3, '2024-03-15', 55.0)
(9, 1, '2024-03-20', 250.0)
(10, 4, '2024-03-22', 75.6)
(11, 1, '2024-01-10', 150.5)
(12, 2, '2024-01-12', 200.0)
(13, 1, '2024-02-05', 45.75)
(14, 3, '2024-02-14', 320.0)
(15, 4, '2024-03-01', 12.99)
(16, 5, '2024-03-05', 89.5)
(17, 2, '2024-03-10', 110.25)
(18, 3, '2024-03-15', 55.0)
(19, 1, '2024-03-20', 250.0)
(20, 4, '2024-03-22', 75.6)
(21, 1, '2024-01-10', 150.5)
(22, 2, '2024-01-12', 200.0)
(23, 1, '2024-02-05', 45.75)
(24, 3, '2024-02-14', 320.0)
(25, 4, '2024-03-01', 12.99)
(26, 5, '2024-03-05', 89.5)
(27, 2, '2024-03-10', 110.25)
(28, 3, '2024-03-15', 55.0)
(29, 1, '2024-03-20', 250.0)
(30, 4, '2024-03-22', 75.6)


<br>
<br>

# Creating Tools

In [50]:
from langchain_community.utilities import SQLDatabase

In [51]:
db = SQLDatabase.from_uri("sqlite:///mydb.db") # SQLite database

In [52]:
db

In [53]:
db.dialect # Version of SQL

'sqlite'

In [54]:
db.get_usable_table_names() # List of the tables created

['customers', 'employees', 'orders']

In [55]:
from langchain_groq import ChatGroq

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant")

In [56]:
llm.invoke("Hello, how are you?") # Invoking the LLM

AIMessage(content="I'm just a language model, so I don't have feelings in the same way that humans do, but thanks for asking. I'm functioning properly and ready to help with any questions or tasks you have. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 41, 'total_tokens': 91, 'completion_time': 0.080984217, 'completion_tokens_details': None, 'prompt_time': 0.006093219, 'prompt_tokens_details': None, 'queue_time': 0.176967234, 'total_time': 0.087077436}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cb4a0-b838-76e3-a768-4fcb269c3b4c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 50, 'total_tokens': 91})

In [57]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [58]:
# Creating the toolkit for all the available tools

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [59]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x113346bd0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x113346bd0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x113346bd0>),
 QuerySQLCheckerTool(description='Use this tool to double check if your 

In [60]:
tools = toolkit.get_tools()

In [61]:
for tool in tools:
  print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [62]:
list_tables_tool = next((tool for tool in tools if tool.name == "sql_db_list_tables"), None)

In [63]:
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x113346bd0>)

In [64]:
# Fetching table schema tool

get_schema_tool = next((tool for tool in tools if tool.name == "sql_db_schema"), None)

In [65]:
get_schema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x113346bd0>)

In [66]:
list_tables_tool.invoke("")

'customers, employees, orders'

In [67]:
print(get_schema_tool.invoke("customers")) # Getting the schema of the table "custoers" + first few rows


CREATE TABLE customers (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
customer_id	first_name	last_name	email	phone
1	John	Doe	john.doe@email.com	555-0101
2	Jane	Smith	jane.smith@email.com	555-0102
3	Michael	Brown	michael.b@email.com	555-0103
*/


In [68]:
# Function for running query

from langchain_core.tools import tool
@tool
def db_query_tool(query: str) -> str:
  """
  Execute a SQL query against the database and return the result.
  If the query is invalid or returns no result, an error message will be returned.
  In case of an error, the user is advised to rewrite the query and try again.
  """
  result = db.run_no_throw(query)
  if not result:
    return "Error : Query Failed ! Please re-write your query and try again."
  return result

In [69]:
db_query_tool.invoke("SELECT * FROM employees;")

"[(1, 'Alice', 'Johnson', 'alice.j@example.com', '2023-01-15', 75000), (2, 'Bob', 'Smith', 'bob.s@example.com', '2023-03-20', 68000), (3, 'Charlie', 'Davis', 'charlie.d@example.com', '2022-11-05', 82000), (4, 'Diana', 'Prince', 'diana.p@example.com', '2024-02-10', 95000), (5, 'Edward', 'Norton', 'edward.n@example.com', '2023-06-30', 70000)]"

In [70]:
db.run("SELECT * FROM employees;")

"[(1, 'Alice', 'Johnson', 'alice.j@example.com', '2023-01-15', 75000), (2, 'Bob', 'Smith', 'bob.s@example.com', '2023-03-20', 68000), (3, 'Charlie', 'Davis', 'charlie.d@example.com', '2022-11-05', 82000), (4, 'Diana', 'Prince', 'diana.p@example.com', '2024-02-10', 95000), (5, 'Edward', 'Norton', 'edward.n@example.com', '2023-06-30', 70000)]"

<br>
<br>

## `query_check`

In [75]:
from typing import Annotated, Literal
from langchain_core.messages import AIMessage
# from langchain_core.pydantic_v1 import BaseModel, Field
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import AnyMessage, add_messages
from typing import Any
from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks
from langgraph.prebuilt import ToolNode

In [76]:
from langchain_core.prompts import ChatPromptTemplate

query_check_system = """You are a SQL expert with a strong attention to detail.
Double check the SQLite query for common mistakes, including:
  - Using NOT IN with NULL values
  - Using UNION when UNION ALL should have been used
  - Using BETWEEN for exclusive ranges
  - Data type mismatch in predicates
  - Properly quoting identifiers
  - Using the correct number of arguments for functions
  - Casting to the correct data type  
  - Using the proper columns for joins

If there are any of the above mistakes, rewrite the query. If there are no mistakes, just reproduce the original query.

You will call the appropriate tool to execute the query after running this check."""

query_check_prompt = ChatPromptTemplate.from_messages ([("system", query_check_system), ("placeholder", "{messages}")])

query_check = query_check_prompt | llm.bind_tools ([db_query_tool])

query_check.invoke({"messages": [("user", "SELECT * FROM employees LIMIT 5;")]})

AIMessage(content='SELECT * FROM employees LIMIT 5 is a valid query, but it will return only a random 5 rows due to the nature of LIMIT without an ORDER BY clause. If you want the first 5 rows, you should use ORDER BY and LIMIT like this:\n\nSELECT * FROM employees ORDER BY id LIMIT 5;\n\nHowever, since we don\'t know the structure of the table, here is the original query with a comment to note its randomness:\n\nSELECT * FROM employees /* returns a random 5 rows */ LIMIT 5;\n\n=function db_query_tool>{"query": "SELECT * FROM employees /* returns a random 5 rows */ LIMIT 5;"}</function>', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 135, 'prompt_tokens': 440, 'total_tokens': 575, 'completion_time': 0.270273635, 'completion_tokens_details': None, 'prompt_time': 0.036455944, 'prompt_tokens_details': None, 'queue_time': 0.066572332, 'total_time': 0.306729579}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_

In [77]:
query_check.invoke({"messages" : [("user", "SELECT * FROM employees LIMIT 5;")]})

AIMessage(content='SELECT * FROM employees LIMIT 5;\n\n', additional_kwargs={'tool_calls': [{'id': 'yjjcfd4nh', 'function': {'arguments': '{"query":"SELECT * FROM employees LIMIT 5;"}', 'name': 'db_query_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 440, 'total_tokens': 469, 'completion_time': 0.042285997, 'completion_tokens_details': None, 'prompt_time': 0.045032893, 'prompt_tokens_details': None, 'queue_time': 0.066512761, 'total_time': 0.08731889}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cb4a3-8bbe-7e31-8733-f4b40384cf3f-0', tool_calls=[{'name': 'db_query_tool', 'args': {'query': 'SELECT * FROM employees LIMIT 5;'}, 'id': 'yjjcfd4nh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 440, 'output_tokens': 29, 'total_tokens': 469})

<br>
<br>

## `query_gen`

In [78]:
class SubmitFinalAnswer(BaseModel):
  """Submit the final answer to the user based on the query results."""
  final_answer: str = Field(..., description="The final answer to the user based on the query results.")

# Add a note for a model to the generate a query based on the user's question and schema
query_gen_note = """You are a SQL expert with a strong attention to detail.
Given an input question, output a syntactically correct SQLite query to run, then look at the results of the query and return the answer.

DO NOT call any tool besides Submit FinalAnswer to submit the final answer.

When generating the query:

Output the SQL query that answers the input question without a tool call.

Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 5 results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

If you get an error while executing a query, rewrite the query and try again.

If you get an empty result set, you should try to rewrite the query to get a non-empty result set.
NEVER make stuff up if you don't have enough information to answer the query... just say you don't have enough information.

If you have enough information to answer the input question, simply invoke the appropriate tool to submit the final answer to the user.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database. Do not return any sql query except answer."""


query_gen_prompt = ChatPromptTemplate.from_messages ([("system", query_gen_note), ("placeholder", "{messages}")])

query_gen = query_gen_prompt | llm.bind_tools ([SubmitFinalAnswer])

<br>
<br>

## Creating Agent